# Model Evaluation: Real vs. Physics-Informed vs. GAN Synthetic Data

This notebook puts every regression model currently in this project
through the same test: **given a
voltammogram signal, how well does the model recover the concentration?**

## Models under test

| Key | Algorithm | Input | Hyperparameter source |
|---|---|---|---|
| `ridge` | Ridge regression | engineered features | `full_range_data_augmentation.ipynb` ablation study |
| `random_forest` | Random Forest | engineered features | ablation study |
| `xgboost` | XGBoost | engineered features | ablation study |
| `svr` | SVR (RBF) | engineered features | `training/tune_ml_hparams.py` (nested LOOCV + `skopt.BayesSearchCV`) |
| `xgboost_tuned` | XGBoost | engineered features | `training/tune_ml_hparams.py` |
| `mlp` | MLP | engineered features | `training/tune_dl_hparams.py` (Optuna) |
| `cnn` | 1D-CNN | **raw 229-point signal** | `training/tune_dl_hparams.py` (Optuna) |

`svr` / `xgboost_tuned` / `mlp`'s hyperparameters selected (after tuning script) specifically for the `experimental` (23-feature) suite. Reusing those same
hyperparameter for the `core`/`extended` suite, being a same-hyperparameters,
different-inputs comparison.
`cnn` has no suite dimension at all, so it always trains on the
raw signal.

## Data sources under test

1. **Physics-informed augmented data** (`raw/raw_signals_augmented.csv`,
   generated by `augmentation_pipeline.py`'s Long & Winefordner noise model +
   PCHIP Ip-concentration spline).
2. **GAN-synthetic data**, both architectures, both training regimes:
   `wgangp` / `timegan` x trained-on-real / trained-on-combined
   (`gan_output/samples_trained_on_*`).
3. **Real data** (`raw/raw_signals_real.csv`).

## Handling data leakage

Every model here is first fit once, on the real 40 signals only
(mirrors `export_models.py`) and then tested against (1) and (2) above. 
Since the physics-informed and GAN data were not seen during that fit, 
those two tests are leakage-free.

Data leakage risk: on the real data.
In `full_range_data_augmentation.ipynb` I solved this with **Protocol C**: hold out one real signal, train on the rest of the real signals + all the physics-augmented ones, test on the held-out real signal, repeat LOOCV-style over all 40. 
Protocol C is implemented in section 6.

## 1 · Imports

In [ ]:
# for root anchor (notebook moved into subfolder)
import sys
from pathlib import Path
ROOT_DIR = Path().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut

import data_registry as dr
from augmentation_pipeline import FEATURE_COLUMNS_BY_SUITE
from model_registry import MODEL_FACTORIES, MODEL_LABELS, RAW_SIGNAL_MODELS, predict, train_models
from plot_style import apply_default_plotly_layout

import paths
SUITES = ['core', 'extended', 'experimental']  # tabular suites; 'cnn' always uses 'raw_signal'
TABULAR_MODEL_KEYS = [k for k in MODEL_FACTORIES if k not in RAW_SIGNAL_MODELS]

# Fixed per-model color, reused across every chart in this notebook
MODEL_COLORS = {
    'ridge': '#17becf', 'random_forest': '#e34a1a', 'xgboost': '#5a23c4',
    'svr': '#2ca02c', 'xgboost_tuned': '#c9a227', 'mlp': '#0b5fa5', 'cnn': '#a83279',
}


2026-07-06 01:41:26.646 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-06 01:41:26.650 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-06 01:41:26.652 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


## 2 · Load every data source

All sources share the same 229-point potential grid and row format
(`concentration`, `I_0` .. `I_228`), so one loader works for all of them.


In [2]:

def load_signal_csv(path: str) -> tuple[np.ndarray, np.ndarray]:
    df = pd.read_csv(path)
    y = df['concentration'].to_numpy(dtype=float)
    X = df.drop(columns=['concentration']).to_numpy(dtype=float)
    return X, y


E = pd.read_csv(paths.POTENTIAL_GRID_CSV)['potential_V'].to_numpy(dtype=float)

X_real, y_real = load_signal_csv(paths.REAL_SIGNALS_CSV)
X_aug, y_aug = load_signal_csv(paths.AUGMENTED_SIGNALS_CSV)

GAN_SOURCE_PATHS = {
    'wgangp_real':      paths.WGANGP_SIGNALS_CSV,
    'wgangp_combined':  paths.WGANGP_SIGNALS_TRAINED_COMBINED_CSV,
    'timegan_real':     paths.TIMEGAN_SIGNALS_CSV,
    'timegan_combined': paths.TIMEGAN_SIGNALS_TRAINED_COMBINED_CSV,
}
gan_data = {name: load_signal_csv(path) for name, path in GAN_SOURCE_PATHS.items()}

print(f'real signals:              {len(y_real)}')
print(f'physics-augmented signals: {len(y_aug)}')
for name, (_, y) in gan_data.items():
    print(f'{name:<18} signals: {len(y)}')


real signals:              40
physics-augmented signals: 300
wgangp_real        signals: 300
wgangp_combined    signals: 300
timegan_real       signals: 300
timegan_combined   signals: 300


## 3 · Featurization and metrics helpers

In [3]:

def featurize(X: np.ndarray, y: np.ndarray, suite: str) -> pd.DataFrame:
    return dr.featurize(E, X, y, suite=suite)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return dict(
        MAE=float(mean_absolute_error(y_true, y_pred)),
        RMSE=float(np.sqrt(mean_squared_error(y_true, y_pred))),
        R2=float(r2_score(y_true, y_pred)),
        n=int(len(y_true)),
    )


## 4 · Fit every model, once, on real data only

Mirrors `export_models.py`: one fit per (model, suite) on all 40 real
signals - not a search, not a nested CV. These are the models Section 5 tests against physics-augmented and GAN data.


In [4]:

def fit_all_models(X: np.ndarray, y: np.ndarray) -> dict:
    # Returns {(model_name, suite): (fitted_model, imputer, feature_columns)}.
    fitted = {}
    for suite in SUITES:
        feat_df = featurize(X, y, suite)
        cols = FEATURE_COLUMNS_BY_SUITE[suite]
        X_feat, y_feat = feat_df[cols], feat_df['concentration']
        models, imputer, _ = train_models(X_feat, y_feat, TABULAR_MODEL_KEYS)
        for name, model in models.items():
            fitted[(name, suite)] = (model, imputer, cols)

    raw_cols = FEATURE_COLUMNS_BY_SUITE['raw_signal']
    raw_feat_df = featurize(X, y, 'raw_signal')
    X_raw, y_raw = raw_feat_df[raw_cols], raw_feat_df['concentration']
    cnn_models, _, raw_imputer = train_models(X_raw, y_raw, ['cnn'], X_raw_signal=X_raw)
    fitted[('cnn', 'raw_signal')] = (cnn_models['cnn'], raw_imputer, raw_cols)
    return fitted


t0 = time.time()
real_fitted = fit_all_models(X_real, y_real)
print(f'Fitted {len(real_fitted)} (model, suite) combinations on {len(y_real)} real signals '
      f'in {time.time() - t0:.1f}s')


Fitted 19 (model, suite) combinations on 40 real signals in 14.3s


## 5 · Test against physics-augmented and GAN data (no leakage)

Every model above was fit on real data alone, so testing it against
synthetic data it never saw is leakage-free by construction.


In [5]:

def evaluate_against(fitted: dict, X_test: np.ndarray, y_test: np.ndarray, test_source: str) -> pd.DataFrame:
    rows = []
    for (model_name, suite), (model, imputer, cols) in fitted.items():
        feat_df = featurize(X_test, y_test, suite)
        X_feat = feat_df[cols]
        y_true = feat_df['concentration'].to_numpy(dtype=float)
        y_pred = predict(model, imputer, X_feat)
        rows.append(dict(model=model_name, suite=suite, test_source=test_source,
                          **regression_metrics(y_true, y_pred)))
    return pd.DataFrame(rows)


no_leak_frames = [evaluate_against(real_fitted, X_aug, y_aug, 'physics_augmented')]
for name, (X, y) in gan_data.items():
    no_leak_frames.append(evaluate_against(real_fitted, X, y, name))

no_leak_results = pd.concat(no_leak_frames, ignore_index=True)
no_leak_results.sort_values(['test_source', 'suite', 'MAE']).reset_index(drop=True)


,model,suite,test_source,MAE,RMSE,R2,n
0,ridge,core,physics_augmented,0.834156,1.653616,0.993346,300
1,random_forest,core,physics_augmented,1.469208,3.454084,0.970968,300
2,xgboost,core,physics_augmented,2.473127,7.111644,0.876930,300
3,xgboost_tuned,core,physics_augmented,2.904095,8.236789,0.834907,300
4,mlp,core,physics_augmented,4.153656,12.503578,0.619564,300
...,...,...,...,...,...,...,...
90,xgboost,extended,wgangp_real,2.417894,5.654298,0.933887,300
91,svr,extended,wgangp_real,2.907294,5.953983,0.926693,300
92,xgboost_tuned,extended,wgangp_real,3.186864,7.326536,0.888998,300
93,mlp,extended,wgangp_real,4.072303,8.098479,0.864375,300


## 6 · Test against real data,  without leakage (Protocol C)

For each of the 40 real signals: fit on the other 39 real signals plus
all physics-augmented ones, predict the held-out signal, repeat LOOCV.
The held-out signal is never part of its own model's training data, so this
is a generalisation estimate on real data.

This runs `n_folds x n_models x n_suites` fits
(40 folds x 6 tabular models x 3 suites + 40 CNN folds).


In [6]:

def protocol_c_real(model_name: str, suite: str) -> dict:
    loo = LeaveOneOut()
    is_raw = model_name in RAW_SIGNAL_MODELS
    cols = FEATURE_COLUMNS_BY_SUITE[suite]
    y_true_all, y_pred_all = [], []

    for train_idx, test_idx in loo.split(X_real):
        X_train = np.concatenate([X_real[train_idx], X_aug], axis=0)
        y_train = np.concatenate([y_real[train_idx], y_aug], axis=0)

        train_feat_df = featurize(X_train, y_train, suite)
        X_feat, y_feat = train_feat_df[cols], train_feat_df['concentration']
        if is_raw:
            models, _, imputer = train_models(X_feat, y_feat, [model_name], X_raw_signal=X_feat)
        else:
            models, imputer, _ = train_models(X_feat, y_feat, [model_name])

        test_feat_df = featurize(X_real[test_idx], y_real[test_idx], suite)
        pred = predict(models[model_name], imputer, test_feat_df[cols])

        y_true_all.append(y_real[test_idx][0])
        y_pred_all.append(pred[0])

    return regression_metrics(np.array(y_true_all), np.array(y_pred_all))


t0 = time.time()
protocol_c_rows = []
for model_name in TABULAR_MODEL_KEYS:
    for suite in SUITES:
        m = protocol_c_real(model_name, suite)
        protocol_c_rows.append(dict(model=model_name, suite=suite, test_source='real_protocol_c', **m))
        print(f'  {model_name:<14} {suite:<12} MAE={m["MAE"]:.3f}  ({time.time() - t0:.0f}s elapsed)')

m = protocol_c_real('cnn', 'raw_signal')
protocol_c_rows.append(dict(model='cnn', suite='raw_signal', test_source='real_protocol_c', **m))
print(f'  {"cnn":<14} {"raw_signal":<12} MAE={m["MAE"]:.3f}  ({time.time() - t0:.0f}s elapsed)')

protocol_c_results = pd.DataFrame(protocol_c_rows)
print(f'\nProtocol C done in {time.time() - t0:.0f}s total')
protocol_c_results.sort_values(['suite', 'MAE']).reset_index(drop=True)


  ridge          core         MAE=1.214  (8s elapsed)
  ridge          extended     MAE=1.172  (44s elapsed)
  ridge          experimental MAE=0.700  (127s elapsed)
  random_forest  core         MAE=0.237  (172s elapsed)
  random_forest  extended     MAE=0.354  (244s elapsed)
  random_forest  experimental MAE=0.402  (366s elapsed)
  xgboost        core         MAE=0.271  (379s elapsed)
  xgboost        extended     MAE=0.297  (418s elapsed)
  xgboost        experimental MAE=0.260  (504s elapsed)
  svr            core         MAE=1.668  (513s elapsed)
  svr            extended     MAE=1.759  (549s elapsed)
  svr            experimental MAE=1.290  (628s elapsed)
  xgboost_tuned  core         MAE=0.193  (643s elapsed)
  xgboost_tuned  extended     MAE=0.216  (683s elapsed)
  xgboost_tuned  experimental MAE=0.174  (774s elapsed)
  mlp            core         MAE=3.645  (936s elapsed)
  mlp            extended     MAE=3.523  (1108s elapsed)
  mlp            experimental MAE=3.574  (1374s el

,model,suite,test_source,MAE,RMSE,R2,n
0,xgboost_tuned,core,real_protocol_c,0.192831,0.463281,0.999714,40
1,random_forest,core,real_protocol_c,0.236746,0.727641,0.999295,40
2,xgboost,core,real_protocol_c,0.271334,0.537308,0.999616,40
3,ridge,core,real_protocol_c,1.213983,2.093987,0.994165,40
4,svr,core,real_protocol_c,1.667922,4.350571,0.974811,40
5,mlp,core,real_protocol_c,3.645355,9.668412,0.875597,40
6,xgboost_tuned,experimental,real_protocol_c,0.174218,0.459803,0.999719,40
7,xgboost,experimental,real_protocol_c,0.260386,0.536147,0.999617,40
8,random_forest,experimental,real_protocol_c,0.401625,0.952329,0.998793,40
9,ridge,experimental,real_protocol_c,0.699735,1.355658,0.997554,40


In [9]:
protocol_c_results.to_csv(paths.RESULTS_DIR / "all_models_protocol_c.csv", index=False)
no_leak_results.to_csv(paths.RESULTS_DIR / "all_models_no_leak.csv", index=False)

## 7 · Combined results table

In [7]:

all_results = pd.concat([no_leak_results, protocol_c_results], ignore_index=True)
all_results['model_label'] = all_results['model'].map(MODEL_LABELS)
all_results.to_csv(paths.MODEL_EVALUATION_RESULTS_CSV, index=False)
all_results.sort_values(['test_source', 'suite', 'MAE']).reset_index(drop=True)


,model,suite,test_source,MAE,RMSE,R2,n,model_label
0,ridge,core,physics_augmented,0.834156,1.653616,0.993346,300,Ridge
1,random_forest,core,physics_augmented,1.469208,3.454084,0.970968,300,Random Forest
2,xgboost,core,physics_augmented,2.473127,7.111644,0.876930,300,XGBoost
3,xgboost_tuned,core,physics_augmented,2.904095,8.236789,0.834907,300,XGBoost (LOOCV-tuned)
4,mlp,core,physics_augmented,4.153656,12.503578,0.619564,300,MLP (tuned)
...,...,...,...,...,...,...,...,...
109,xgboost,extended,wgangp_real,2.417894,5.654298,0.933887,300,XGBoost
110,svr,extended,wgangp_real,2.907294,5.953983,0.926693,300,"SVR (RBF, tuned)"
111,xgboost_tuned,extended,wgangp_real,3.186864,7.326536,0.888998,300,XGBoost (LOOCV-tuned)
112,mlp,extended,wgangp_real,4.072303,8.098479,0.864375,300,MLP (tuned)


## 8 · Visualisation

Two comparisons, both on the `experimental` suite (the one every tuned
model's hyperparameters actually target):

1. **MAE by test source** - how does each model's error change across
   physics-augmented data, each GAN variant, and the leakage-free real-data
   read (Protocol C)?
2. **MAE by feature suite** - does the extra feature complexity in
   `extended`/`experimental` actually help, for a fixed test source?


In [10]:

def plot_mae_by_test_source(df: pd.DataFrame, suite: str = 'experimental') -> go.Figure:
    sub = df[(df['suite'] == suite) | (df['model'] == 'cnn')]
    test_sources = list(dict.fromkeys(sub['test_source']))  # stable order, de-duped

    fig = go.Figure()
    for model_name, label in MODEL_LABELS.items():
        row = sub[sub['model'] == model_name].set_index('test_source').reindex(test_sources)
        fig.add_trace(go.Bar(x=test_sources, y=row['MAE'], name=label,
                              marker_color=MODEL_COLORS[model_name]))
    fig.update_layout(barmode='group')
    return apply_default_plotly_layout(
        fig, title_text=f'MAE by test source ({suite} suite, cnn always raw signal)',
        xaxis_title='Test source', yaxis_title='MAE (uM)')


def plot_mae_by_suite(df: pd.DataFrame, test_source: str = 'physics_augmented') -> go.Figure:
    sub = df[(df['test_source'] == test_source) & (df['model'] != 'cnn')]

    fig = go.Figure()
    for model_name in TABULAR_MODEL_KEYS:
        row = sub[sub['model'] == model_name].set_index('suite').reindex(SUITES)
        fig.add_trace(go.Bar(x=SUITES, y=row['MAE'], name=MODEL_LABELS[model_name],
                              marker_color=MODEL_COLORS[model_name]))
    fig.update_layout(barmode='group')
    return apply_default_plotly_layout(
        fig, title_text=f'MAE by feature suite - {test_source}',
        xaxis_title='Feature suite', yaxis_title='MAE (uM)')


plot_mae_by_test_source(all_results, suite='experimental').show()


In [11]:

plot_mae_by_suite(all_results, test_source='physics_augmented').show()


## 9 · Discussion




## 10 · Physics-informed vs. GAN augmentation: which is more realistic

Section 5 already gives us what we need for this: every model in
`real_fitted` was trained on real data only, then scored against each
synthetic source using *that source's own labeled concentration*. Because
none of these models ever saw the synthetic data, the resulting error is
 a train-on-real / test-on-synthetic (TRTS) fidelity check. MAE is in the same units (uM) for every source, so it's the metric to compare across sources. R2 is relative to each source's own label variance and is only safely compared *within* a source or between sources with matching label distributions.


In [12]:
def describe_y(y: np.ndarray) -> dict:
    return dict(n=len(y), min=float(y.min()), max=float(y.max()),
                mean=float(y.mean()), std=float(y.std()))


dist_rows = [
    dict(source='real', **describe_y(y_real)),
    dict(source='physics_augmented', **describe_y(y_aug)),
]
for name, (_, y) in gan_data.items():
    dist_rows.append(dict(source=name, **describe_y(y)))

pd.DataFrame(dist_rows)


,source,n,min,max,mean,std
0,real,40,0.100000,100.000000,16.243750,27.411955
1,physics_augmented,300,0.101904,98.457009,10.295141,20.271858
2,wgangp_real,300,0.103558,93.360152,14.362122,21.990436
3,wgangp_combined,300,0.103558,93.360152,14.362122,21.990436
4,timegan_real,300,0.103558,93.360152,14.362122,21.990436
5,timegan_combined,300,0.103558,93.360152,14.362122,21.990436


The four GAN sources share an *identical* concentration template
(min 0.10, max 93.36, mean 14.36, std 21.99 uM - to the decimal) - only the
generated I(E) signal differs by architecture/training-regime, so any
comparison among `wgangp_real` / `wgangp_combined` / `timegan_real` /
`timegan_combined` is fully apples-to-apples. `physics_augmented`'s
distribution is close but not identical (mean 10.30, std 20.27 uM) - a
narrower spread than the GAN template and than the real 40-signal
distribution it was built from (mean 16.24, std 27.41). It's close enough
to compare, but treat the physics-informed vs. GAN comparison as
indicative rather than exact; the GAN-family comparisons below are exact.

In [13]:
SYNTHETIC_SOURCES = ['physics_augmented', 'wgangp_real', 'wgangp_combined',
                      'timegan_real', 'timegan_combined']

synthetic_results = all_results[all_results['test_source'].isin(SYNTHETIC_SOURCES)]
# experimental suite (what every tuned model targets) + cnn (no suite dimension)
synthetic_exp = synthetic_results[(synthetic_results['suite'] == 'experimental') |
                                   (synthetic_results['model'] == 'cnn')]

fidelity_summary = (synthetic_exp.groupby('test_source')[['MAE', 'RMSE', 'R2']]
                     .agg(['mean', 'median']).reindex(SYNTHETIC_SOURCES))
fidelity_summary


MAE                 RMSE                  R2          
                       mean    median       mean    median      mean    median
test_source                                                                   
physics_augmented  3.048266  2.056689   7.255546  5.570506  0.807125  0.924490
wgangp_real        6.073212  2.433580  13.534058  5.433275  0.118926  0.939429
wgangp_combined    4.281116  3.738027   8.854090  8.347647  0.800343  0.855901
timegan_real       3.475853  3.046307   6.223636  4.713340  0.884171  0.954060
timegan_combined   4.241447  4.042094   7.793579  7.571610  0.816704  0.889582

In [14]:
SOURCE_COLORS = {
    'physics_augmented': '#5a5a5a', 'wgangp_real': '#2ca02c',
    'wgangp_combined': '#98df8a', 'timegan_real': '#1f77b4',
    'timegan_combined': '#aec7e8',
}


def plot_median_mae_by_source(df: pd.DataFrame) -> go.Figure:
    med = df.groupby('test_source')['MAE'].median().reindex(SYNTHETIC_SOURCES)
    fig = go.Figure(go.Bar(x=SYNTHETIC_SOURCES, y=med.values,
                            marker_color=[SOURCE_COLORS[s] for s in SYNTHETIC_SOURCES]))
    return apply_default_plotly_layout(
        fig, title_text='Median MAE by augmentation source (real-fitted models, experimental suite + cnn)',
        xaxis_title='Augmentation source', yaxis_title='Median MAE (uM)')


plot_median_mae_by_source(synthetic_exp).show()


### Conclusion

